# Module 1 — Medallion Architecture (Bronze / Silver / Gold)
Exam domain: **Data Modeling** · Interview weight: ⭐⭐⭐⭐⭐

Runs standalone in Google Colab — no Databricks account needed.

In [ ]:
# 1. Install PySpark + Delta Lake
!pip install -q pyspark==3.5.1 delta-spark==3.2.0

In [ ]:
# 2. Create Spark session with Delta support
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (SparkSession.builder
    .appName("Module1-Medallion")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark

## Bronze — raw ingestion
Load raw data as-is, add ingestion metadata. No transformations.

In [ ]:
from pyspark.sql import functions as F

raw_data = [
    (1, "Alice", "2024-01-01", "100.5"),
    (2, "Bob", "2024-01-02", "bad_value"),
    (3, None, "2024-01-03", "200.0"),
]
raw_df = spark.createDataFrame(raw_data, ["id", "name", "event_date", "amount_raw"])

bronze_df = raw_df.withColumn("ingestion_ts", F.current_timestamp()) \
                   .withColumn("source_file", F.lit("colab_demo"))

bronze_df.write.format("delta").mode("overwrite").save("/content/lake/bronze/events")
bronze_df.show()

## Silver — cleaned & validated
Cast types, drop bad rows, enforce schema/quality rules.

In [ ]:
bronze_df = spark.read.format("delta").load("/content/lake/bronze/events")

silver_df = (bronze_df
    .withColumn("amount", F.col("amount_raw").cast("double"))
    .filter(F.col("amount").isNotNull())
    .filter(F.col("name").isNotNull())
    .withColumn("event_date", F.to_date("event_date"))
    .drop("amount_raw", "source_file"))

silver_df.write.format("delta").mode("overwrite").save("/content/lake/silver/events")
silver_df.show()

## Gold — business-level aggregates
Ready for BI / reporting.

In [ ]:
silver_df = spark.read.format("delta").load("/content/lake/silver/events")

gold_df = (silver_df
    .groupBy("event_date")
    .agg(F.sum("amount").alias("total_amount"),
         F.count("*").alias("num_events")))

gold_df.write.format("delta").mode("overwrite").save("/content/lake/gold/daily_summary")
gold_df.show()

## Interview questions to review
1. Why separate Bronze/Silver/Gold instead of one clean table?
2. What belongs in Bronze vs Silver validation logic?
3. How would you handle schema drift arriving in Bronze?
4. Why use Delta format instead of plain Parquet at each layer?